In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir -p squad
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json -O squad/train-v2.0.json
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json -O squad/dev-v2.0.json

--2026-05-29 22:04:37--  https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42123633 (40M) [application/json]
Saving to: ‘squad/train-v2.0.json’

squad/train-v2.0.js 100%[===================>]  40.17M   195MB/s    in 0.2s    

2026-05-29 22:04:38 (195 MB/s) - ‘squad/train-v2.0.json’ saved [42123633/42123633]

--2026-05-29 22:04:38--  https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4370528 (4.2M) [application/json]
Saving to: ‘squad/dev-v2.0.json’

squ

In [3]:
!pip install -U accelerate

In [4]:
!pip install transformers datasets evaluate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [5]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained(
    "bert-base-uncased"
)

# =========================
# READ DATA
# =========================

import json

def read_squad(path):

    with open(path, 'rb') as f:
        squad_dict = json.load(f)

    contexts = []
    questions = []
    answers = []

    for group in squad_dict['data']:

        for passage in group['paragraphs']:

            context = passage['context']

            for qa in passage['qas']:

                question = qa['question']

                # impossible question
                if qa["is_impossible"]:

                    answer = {
                        "text": "",
                        "answer_start": -1,
                        "answer_end": -1,
                    }

                else:

                    first_answer = qa['answers'][0]

                    answer = {
                        "text": first_answer['text'],
                        "answer_start": first_answer['answer_start'],
                    }

                contexts.append(context)
                questions.append(question)
                answers.append(answer)

    return contexts, questions, answers


train_contexts, train_questions, train_answers = read_squad(
    "/content/squad/train-v2.0.json"
)

val_contexts, val_questions, val_answers = read_squad(
    "/content/squad/dev-v2.0.json"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
# =========================
# FIX ANSWER END INDEX
# =========================

def add_end_idx(answers, contexts):

    for answer, context in zip(answers, contexts):

        # impossible question
        if answer["answer_start"] == -1:
            continue

        gold_text = answer["text"]

        start_idx = answer["answer_start"]

        end_idx = start_idx + len(gold_text)

        # sometimes SQuAD labels are off by 1-2 chars
        if context[start_idx:end_idx] == gold_text:

            answer["answer_end"] = end_idx

        elif context[start_idx - 1:end_idx - 1] == gold_text:

            answer["answer_start"] = start_idx - 1
            answer["answer_end"] = end_idx - 1

        elif context[start_idx - 2:end_idx - 2] == gold_text:

            answer["answer_start"] = start_idx - 2
            answer["answer_end"] = end_idx - 2

        else:

            answer["answer_end"] = end_idx


add_end_idx(train_answers, train_contexts)
add_end_idx(val_answers, val_contexts)

In [7]:
# =========================
# TOKENIZE
# =========================

train_encodings = tokenizer(
    train_questions,
    train_contexts,
    truncation="only_second",
    max_length=384,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding=False,
)

val_encodings = tokenizer(
    val_questions,
    val_contexts,
    truncation="only_second",
    max_length=384,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding=False,
)

# keep a deep copy of val_encodings for postprocessing (offset_mapping will be popped later)
import copy
val_encodings_for_postprocessing = copy.deepcopy(val_encodings)

In [8]:
# =========================
# TOKEN POSITIONS
# =========================

def add_token_positions(encodings, answers):

    start_positions = []
    end_positions = []

    sample_mapping = encodings["overflow_to_sample_mapping"]
    offset_mapping = encodings["offset_mapping"]

    for i, offsets in enumerate(offset_mapping):

        input_ids = encodings["input_ids"][i]

        cls_index = input_ids.index(
            tokenizer.cls_token_id
        )

        sample_idx = sample_mapping[i]

        answer = answers[sample_idx]

        # impossible question
        if answer["answer_start"] == -1:

            start_positions.append(cls_index)
            end_positions.append(cls_index)

            continue

        start_char = answer["answer_start"]
        end_char = answer["answer_end"]

        sequence_ids = encodings.sequence_ids(i)

        # context start/end
        token_start_index = 0

        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1

        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # answer outside current span
        if not (
            offsets[token_start_index][0] <= start_char
            and
            offsets[token_end_index][1] >= end_char
        ):

            start_positions.append(cls_index)
            end_positions.append(cls_index)

        else:

            # start token
            while (
                token_start_index < len(offsets)
                and
                offsets[token_start_index][0] <= start_char
            ):
                token_start_index += 1

            start_positions.append(
                token_start_index - 1
            )

            # end token
            while (
                offsets[token_end_index][1] >= end_char
            ):
                token_end_index -= 1

            end_positions.append(
                token_end_index + 1
            )

    encodings.update({
        "start_positions": start_positions,
        "end_positions": end_positions,
    })

    encodings.pop("offset_mapping")

add_token_positions(
    train_encodings,
    train_answers
)

add_token_positions(
    val_encodings,
    val_answers
)


In [9]:
# =========================
# CHECK IMPOSSIBLE COUNT
# =========================

count = 0
count_full = 0

for i in range(len(train_encodings["start_positions"])):

    input_ids = train_encodings["input_ids"][i]

    cls_index = input_ids.index(
        tokenizer.cls_token_id
    )

    count_full += 1

    if (
        train_encodings["start_positions"][i] == cls_index
        and
        train_encodings["end_positions"][i] == cls_index
    ):
        count += 1

print("Impossible samples:", count)
print("Samples:", count_full)

Impossible samples: 44748
Samples: 131754


In [10]:
import torch
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from evaluate import load
import numpy as np

# =========================
# DATASET
# =========================

class SquadDataset(torch.utils.data.Dataset):

    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):

        return {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

    def __len__(self):

        return len(self.encodings["input_ids"])


train_dataset = SquadDataset(train_encodings)
val_dataset = SquadDataset(val_encodings)

In [11]:
from transformers import BertForQuestionAnswering
model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [12]:
# =========================
# DYNAMIC PADDING
# =========================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

# =========================
# METRIC
# =========================

metric = load("squad_v2")

In [13]:
# =========================
# METRIC
# =========================

metric = load("squad_v2")

# =========================
# POSTPROCESS & COMPUTE METRICS
# =========================

from collections import OrderedDict, defaultdict
import numpy as np

def postprocess_qa_predictions(examples, features, predictions, tokenizer,
                               n_best_size=20, max_answer_length=30,
                               version_2_with_negative=True, null_score_diff_threshold=0.0):
    all_start_logits, all_end_logits = predictions
    all_start_logits = np.array(all_start_logits)
    all_end_logits = np.array(all_end_logits)

    # map example id -> list of feature indices
    features_per_example = defaultdict(list)
    for i, f in enumerate(features):
        features_per_example[f["example_id"]].append(i)

    final_predictions = OrderedDict()

    for example in examples:
        example_id = example["id"]
        context = example["context"]
        feature_indices = features_per_example[example_id]

        prelim_predictions = []
        min_null_score = None

        for fi in feature_indices:
            start_logits = all_start_logits[fi]
            end_logits = all_end_logits[fi]
            offset_mapping = features[fi]["offset_mapping"]
            input_ids = features[fi]["input_ids"]

            # CLS score for null
            try:
                cls_index = input_ids.index(tokenizer.cls_token_id)
            except ValueError:
                cls_index = 0
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or feature_null_score < min_null_score:
                min_null_score = feature_null_score

            start_indexes = np.argsort(start_logits)[-1: -n_best_size-1: -1]
            end_indexes = np.argsort(end_logits)[-1: -n_best_size-1: -1]

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offset_mapping) or end_index >= len(offset_mapping):
                        continue
                    if offset_mapping[start_index] is None or offset_mapping[end_index] is None:
                        continue
                    # skip tokens that are not part of the context (offset (0,0))
                    if offset_mapping[start_index][0] == 0 and offset_mapping[start_index][1] == 0:
                        continue
                    if end_index < start_index or (end_index - start_index + 1) > max_answer_length:
                        continue
                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    score = float(start_logits[start_index] + end_logits[end_index])
                    prelim_predictions.append({
                        "offsets": (start_char, end_char),
                        "score": score,
                        "start_logit": float(start_logits[start_index]),
                        "end_logit": float(end_logits[end_index]),
                    })

        if len(prelim_predictions) == 0:
            final_predictions[example_id] = ""
            continue

        prelim_predictions = sorted(prelim_predictions, key=lambda x: x["score"], reverse=True)

        nbest = []
        seen_texts = set()
        for pred in prelim_predictions[:n_best_size]:
            start_char, end_char = pred["offsets"]
            text = context[start_char:end_char]
            if text in seen_texts:
                continue
            seen_texts.add(text)
            nbest.append({"text": text, "score": pred["score"],
                          "start_logit": pred["start_logit"], "end_logit": pred["end_logit"]})

        if version_2_with_negative:
            if min_null_score is None:
                min_null_score = 0.0
            nbest.append({"text": "", "score": float(min_null_score), "start_logit": 0.0, "end_logit": 0.0})

        nbest = sorted(nbest, key=lambda x: x["score"], reverse=True)

        best_non_null_score = next((entry["score"] for entry in nbest if entry["text"] != ""), -1e8)
        if version_2_with_negative and min_null_score > best_non_null_score + null_score_diff_threshold:
            final_predictions[example_id] = ""
        else:
            final_predictions[example_id] = nbest[0]["text"] if nbest[0]["text"] != "" else (nbest[1]["text"] if len(nbest) > 1 else "")

    return final_predictions

def compute_metrics(eval_pred):
    start_logits, end_logits = eval_pred.predictions

    # Build features from saved copy for postprocessing
    features = []
    for i in range(len(val_encodings_for_postprocessing["input_ids"])):
        features.append({
            "input_ids": val_encodings_for_postprocessing["input_ids"][i],
            "offset_mapping": val_encodings_for_postprocessing["offset_mapping"][i],
            "example_id": str(val_encodings_for_postprocessing["overflow_to_sample_mapping"][i]),
        })

    # Build examples list
    examples = []
    for i, (c, q, a) in enumerate(zip(val_contexts, val_questions, val_answers)):
        if a.get("answer_start", -1) == -1:
            answers = {"text": [], "answer_start": []}
        else:
            answers = {"text": [a["text"]], "answer_start": [a["answer_start"]]}
        examples.append({"id": str(i), "question": q, "context": c, "answers": answers})

    final_predictions = postprocess_qa_predictions(
        examples, features, (start_logits, end_logits), tokenizer,
        n_best_size=20, max_answer_length=30,
        version_2_with_negative=True, null_score_diff_threshold=1.0
    )

    formatted_predictions = [
        {"id": k, "prediction_text": v, "no_answer_probability": 0.0}
        for k, v in final_predictions.items()
    ]

    references = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]

    results = metric.compute(predictions=formatted_predictions, references=references)
    return {"f1": results["f1"], "exact_match": results["exact"]}

In [14]:
# =========================
# TRAINING ARGS
# =========================

training_args = TrainingArguments(

    output_dir='./results',

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,

    logging_dir='./logs',
    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",

    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
# =========================
# TRAINER
# =========================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    # dynamic padding
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# =========================
# TRAIN
# =========================

trainer.train()

Epoch,Training Loss,Validation Loss,F1,Exact Match
1,1.086535,1.046609,69.815450,63.185379
2,0.720629,1.221624,69.306446,62.233639
3,0.299367,1.451448,71.371656,64.364525


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=24705, training_loss=0.7845179297909083, metrics={'train_runtime': 7121.0306, 'train_samples_per_second': 55.506, 'train_steps_per_second': 3.469, 'total_flos': 6.174568212804905e+16, 'train_loss': 0.7845179297909083, 'epoch': 3.0})

In [16]:
# =========================
# FINAL EVAL
# =========================

final_results = trainer.evaluate()

print("\n===== FINAL RESULTS =====")

print(
    f"Final F1 Score: "
    f"{final_results['eval_f1']:.2f}"
)

print(
    f"Final Exact Match: "
    f"{final_results['eval_exact_match']:.2f}"
)


===== FINAL RESULTS =====
Final F1 Score: 71.37
Final Exact Match: 64.36


In [17]:
trainer.save_model("/content/drive/MyDrive/bert_qa_model")
tokenizer.save_pretrained("/content/drive/MyDrive/bert_qa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/bert_qa_model/tokenizer_config.json',
 '/content/drive/MyDrive/bert_qa_model/tokenizer.json')

In [46]:
from transformers import BertTokenizerFast
from transformers import BertForQuestionAnswering
import torch
import numpy as np

# Load model đã train
model_path = "/content/drive/MyDrive/bert_qa_model"

tokenizer = BertTokenizerFast.from_pretrained(model_path)
model = BertForQuestionAnswering.from_pretrained(model_path)

model.eval()

# Data mới hoàn toàn

context = """One of the first Norman mercenaries to serve as a Byzantine general was Hervé in the 1050s. By then however, there were already Norman mercenaries serving as far away as Trebizond and Georgia. They were based at Malatya and Edessa, under the Byzantine duke of Antioch, Isaac Komnenos. In the 1060s, Robert Crispin led the Normans of Edessa against the Turks. Roussel de Bailleul even tried to carve out an independent state in Asia Minor with support from the local population, but he was stopped by the Byzantine general Alexius Komnenos."""

question = "When did Herve go up against the Turks?"

def predict(question, context, null_score_diff_threshold=1.0):
    """Tokenize in the same order as preprocessing: question, context.
    Returns predicted answer span (or 'No answer')."""
    # Tokenize (question first, context second) to match training preprocessing
    inputs = tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=384,
        return_offsets_mapping=True,
        padding=True,
        return_tensors="pt",
    )

    # Move tensors to model device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Inference
    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0].cpu()
    end_logits = outputs.end_logits[0].cpu()

    input_ids = inputs["input_ids"][0].cpu().tolist()
    token_type_ids = inputs["token_type_ids"][0].cpu().tolist()
    offsets = inputs["offset_mapping"][0].cpu().tolist()

    # find CLS index
    try:
        cls_index = input_ids.index(tokenizer.cls_token_id)
    except ValueError:
        cls_index = 0

    # best token indices
    start_idx = int(torch.argmax(start_logits).item())
    end_idx = int(torch.argmax(end_logits).item())

    # scores
    best_span_score = (start_logits[start_idx] + end_logits[end_idx]).item()
    no_answer_score = (start_logits[cls_index] + end_logits[cls_index]).item()

    # decide no-answer using same threshold as compute_metrics/postprocess
    if no_answer_score > best_span_score + null_score_diff_threshold:
        return "No answer"

    # ensure indices in right order
    if end_idx < start_idx:
        end_idx = start_idx

    # ensure predicted tokens are inside context (token_type_id == 1)
    if token_type_ids[start_idx] != 1 or token_type_ids[end_idx] != 1:
        return "No answer"

    # map token indices to character positions in context using offsets
    start_char = offsets[start_idx][0]
    end_char = offsets[end_idx][1]

    answer = context[start_char:end_char].strip()
    if answer == "":
        return "No answer"

    return answer

print("Context:", context)
print("Question:", question)
print("Answer:", predict(question, context))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Context: One of the first Norman mercenaries to serve as a Byzantine general was Hervé in the 1050s. By then however, there were already Norman mercenaries serving as far away as Trebizond and Georgia. They were based at Malatya and Edessa, under the Byzantine duke of Antioch, Isaac Komnenos. In the 1060s, Robert Crispin led the Normans of Edessa against the Turks. Roussel de Bailleul even tried to carve out an independent state in Asia Minor with support from the local population, but he was stopped by the Byzantine general Alexius Komnenos.
Question: When did Herve go up against the Turks?
Answer: No answer
